In [1]:
import sys, os

sys.path.insert(0, "/home/ang/usd_ang/lib/python")
os.environ["LD_LIBRARY_PATH"] = "/home/ang/usd_ang/lib"

from pxr import Usd, UsdGeom, Sdf, Gf
import numpy as np


In [2]:
import os
import numpy as np
from pxr import Usd, UsdGeom, Gf


def extract_component_meshes(component_path):
    component_path = os.path.abspath(component_path)

    print("Opening:", component_path)
    print("Exists:", os.path.exists(component_path))

    stage = Usd.Stage.Open(component_path)
    if not stage:
        raise RuntimeError(f"Failed to open {component_path}")

    all_points = []
    all_triangles = []
    vertex_offset = 0

    for prim in stage.Traverse():
        if prim.IsA(UsdGeom.Mesh):
            mesh = UsdGeom.Mesh(prim)

            pts = mesh.GetPointsAttr().Get()
            counts = mesh.GetFaceVertexCountsAttr().Get()
            indices = mesh.GetFaceVertexIndicesAttr().Get()

            if pts is None or counts is None or indices is None:
                continue

            pts_np = np.array([[p[0], p[1], p[2]] for p in pts], dtype=float)
            all_points.append(pts_np)

            idx = 0
            for c in counts:
                if c == 3:
                    tri = [
                        indices[idx + 0] + vertex_offset,
                        indices[idx + 1] + vertex_offset,
                        indices[idx + 2] + vertex_offset,
                    ]
                    all_triangles.append(tri)

                elif c == 4:
                    # triangulate quad
                    a = indices[idx + 0] + vertex_offset
                    b = indices[idx + 1] + vertex_offset
                    c2 = indices[idx + 2] + vertex_offset
                    d = indices[idx + 3] + vertex_offset
                    all_triangles.append([a, b, c2])
                    all_triangles.append([a, c2, d])

                else:
                    # ignore n-gons for now
                    pass

                idx += c

            vertex_offset += len(pts_np)

    if not all_points:
        raise RuntimeError(f"No meshes found in {component_path}")

    points = np.vstack(all_points)
    triangles = np.array(all_triangles, dtype=int)

    return points, triangles


# ------------------------------------------------------------
# Bounds helper
# ------------------------------------------------------------
def compute_bounds(points):
    mins = points.min(axis=0)
    maxs = points.max(axis=0)
    size = maxs - mins
    return mins, maxs, size


# ------------------------------------------------------------
# Exact ray-triangle intersection (Möller–Trumbore)
# Ray origin + t * ray_dir
# ------------------------------------------------------------
def ray_triangle_intersect(ray_origin, ray_dir, v0, v1, v2, eps=1e-9):
    edge1 = v1 - v0
    edge2 = v2 - v0

    h = np.cross(ray_dir, edge2)
    a = np.dot(edge1, h)

    if -eps < a < eps:
        return None  # parallel

    f = 1.0 / a
    s = ray_origin - v0
    u = f * np.dot(s, h)
    if u < 0.0 or u > 1.0:
        return None

    q = np.cross(s, edge1)
    v = f * np.dot(ray_dir, q)
    if v < 0.0 or u + v > 1.0:
        return None

    t = f * np.dot(edge2, q)
    if t > eps:
        return ray_origin + t * ray_dir

    return None


# ------------------------------------------------------------
# Exact terrain height from moon mesh using a vertical ray
# x,z are in MOON LOCAL coordinates
# ------------------------------------------------------------
def terrain_height_at(points, triangles, x, z):
    ray_origin = np.array([x, points[:, 1].max() + 10.0, z], dtype=float)
    ray_dir = np.array([0.0, -1.0, 0.0], dtype=float)

    best_y = None

    for tri in triangles:
        v0 = points[tri[0]]
        v1 = points[tri[1]]
        v2 = points[tri[2]]

        hit = ray_triangle_intersect(ray_origin, ray_dir, v0, v1, v2)
        if hit is not None:
            y = hit[1]
            if best_y is None or y > best_y:
                best_y = y

    if best_y is None:
        raise RuntimeError(f"No terrain hit at x={x}, z={z}")

    return best_y


# ------------------------------------------------------------
# Lowest point of lander geometry in its local frame
# ------------------------------------------------------------
def lander_bottom_y(lander_points):
    return float(lander_points[:, 1].min())


# ------------------------------------------------------------
# Assemble scene:
# - Moon fixed
# - Lander placed relative to moon
# - Exact vertical terrain contact
# - Auto lander scale to fit the tile
# ------------------------------------------------------------
def assemble_scene(
    moon_component,
    lander_component,
    output="lunar_scene.usda",
    moon_position=(0.0, 0.0, 0.0),
    lander_xz=(0.0, 0.0),
    lander_scale=None,
    lander_fraction_of_moon=0.35,
):
    moon_component = os.path.abspath(moon_component)
    lander_component = os.path.abspath(lander_component)
    output = os.path.abspath(output)

    # --------------------------------------------------------
    # Load moon and lander geometry
    # --------------------------------------------------------
    moon_points, moon_triangles = extract_component_meshes(moon_component)
    lander_points, lander_triangles = extract_component_meshes(lander_component)

    moon_min, moon_max, moon_size = compute_bounds(moon_points)
    lander_min, lander_max, lander_size = compute_bounds(lander_points)

    print("\nMoon bounds:")
    print("  min:", moon_min)
    print("  max:", moon_max)
    print("  size:", moon_size)

    print("\nLander bounds (unscaled):")
    print("  min:", lander_min)
    print("  max:", lander_max)
    print("  size:", lander_size)

    # --------------------------------------------------------
    # Auto scale so lander fits the moon tile
    # --------------------------------------------------------
    if lander_scale is None:
        moon_span = min(moon_size[0], moon_size[2])         # smaller horizontal span of moon tile
        lander_span = max(lander_size[0], lander_size[2])   # larger horizontal span of lander
        lander_scale = (lander_fraction_of_moon * moon_span) / lander_span

    print("\nUsing lander_scale =", lander_scale)

    # --------------------------------------------------------
    # Terrain contact
    # lander_xz is RELATIVE TO MOON LOCAL FRAME
    # --------------------------------------------------------
    local_x, local_z = lander_xz
    terrain_y_local = terrain_height_at(moon_points, moon_triangles, local_x, local_z)

    # scaled lander bottom
    bottom_y_local = lander_bottom_y(lander_points) * lander_scale

    # world placement
    moon_tx, moon_ty, moon_tz = moon_position
    lander_world_x = moon_tx + local_x
    lander_world_z = moon_tz + local_z
    lander_world_y = moon_ty + terrain_y_local - bottom_y_local

    print("\nPlacement:")
    print("  terrain_y_local :", terrain_y_local)
    print("  lander_bottom_y :", bottom_y_local)
    print("  lander_world    :", (lander_world_x, lander_world_y, lander_world_z))

    # --------------------------------------------------------
    # Create assembly stage
    # --------------------------------------------------------
    stage = Usd.Stage.CreateNew(output)
    UsdGeom.SetStageUpAxis(stage, UsdGeom.Tokens.y)
    UsdGeom.SetStageMetersPerUnit(stage, 1.0)

    world = UsdGeom.Xform.Define(stage, "/World")
    stage.SetDefaultPrim(world.GetPrim())

    # Moon
    moon = UsdGeom.Xform.Define(stage, "/World/Moon")
    moon.GetPrim().GetReferences().AddReference(moon_component)
    moon.AddTranslateOp().Set(Gf.Vec3d(*moon_position))

    # Lander
    lander = UsdGeom.Xform.Define(stage, "/World/Lander")
    lander.GetPrim().GetReferences().AddReference(lander_component)
    lander.AddScaleOp().Set(Gf.Vec3f(lander_scale, lander_scale, lander_scale))
    lander.AddTranslateOp().Set(Gf.Vec3d(lander_world_x, lander_world_y, lander_world_z))

    stage.GetRootLayer().Save()
    print("\nScene written to:", output)

In [3]:
# assemble_scene(
#     moon_component="../assets/moon/curated/moon_component.usda",
#     lander_component="../assets/lander/curated_lander/lander_component.usda",
#     output="lunar_scene.usda",
#     moon_position=(0.0, 0.0, 0.0),
#     lander_xz=(1, 1),      # relative to moon local frame
#     lander_scale=None,         # auto-compute from moon tile size
#     lander_fraction_of_moon=0.10
# )

In [4]:
import os
import numpy as np
from pxr import Usd, UsdGeom, Gf


# ------------------------------------------------------------
# Asset helper
# ------------------------------------------------------------
class Asset:
    def __init__(self, name, component_path, xz=(0,0), scale=None):
        self.name = name
        self.component_path = os.path.abspath(component_path)
        self.xz = xz
        self.scale = scale


# ------------------------------------------------------------
# General asset placement on terrain
# ------------------------------------------------------------
def place_asset_on_terrain(asset, moon_points, moon_triangles, moon_position):

    points, triangles = extract_component_meshes(asset.component_path)

    mins, maxs, size = compute_bounds(points)

    if asset.scale is None:
        asset.scale = 1.0

    x, z = asset.xz

    terrain_y = terrain_height_at(moon_points, moon_triangles, x, z)

    bottom = lander_bottom_y(points) * asset.scale

    moon_tx, moon_ty, moon_tz = moon_position

    world_x = moon_tx + x
    world_z = moon_tz + z
    world_y = moon_ty + terrain_y - bottom

    return world_x, world_y, world_z, asset.scale


# ------------------------------------------------------------
# Assemble full scene
# ------------------------------------------------------------
def assemble_scene_full(
    moon_component,
    assets,
    output="lunar_scene.usda",
    moon_position=(0,0,0)
):

    moon_component = os.path.abspath(moon_component)
    output = os.path.abspath(output)

    moon_points, moon_triangles = extract_component_meshes(moon_component)

    print("Moon mesh loaded:", len(moon_points), "vertices")

    if os.path.exists(output):
        os.remove(output)

    stage = Usd.Stage.CreateNew(output)
    UsdGeom.SetStageUpAxis(stage, UsdGeom.Tokens.y)
    UsdGeom.SetStageMetersPerUnit(stage, 1.0)

    world = UsdGeom.Xform.Define(stage, "/World")
    stage.SetDefaultPrim(world.GetPrim())

    # --------------------------------------------------------
    # Moon
    # --------------------------------------------------------
    moon = UsdGeom.Xform.Define(stage, "/World/Moon")
    moon.GetPrim().GetReferences().AddReference(moon_component)
    moon.AddTranslateOp().Set(Gf.Vec3d(*moon_position))

    # --------------------------------------------------------
    # Other assets
    # --------------------------------------------------------
    for asset in assets:

        print("\nPlacing:", asset.name)

        x,y,z,scale = place_asset_on_terrain(
            asset,
            moon_points,
            moon_triangles,
            moon_position
        )

        print("  world position:", (x,y,z))
        print("  scale:", scale)

        node = UsdGeom.Xform.Define(stage, f"/World/{asset.name}")

        node.GetPrim().GetReferences().AddReference(asset.component_path)

        node.AddScaleOp().Set(Gf.Vec3f(scale, scale, scale))
        node.AddTranslateOp().Set(Gf.Vec3d(x,y,z))

    stage.GetRootLayer().Save()

    print("\nScene written to:", output)

In [6]:
assets = [

    Asset(
        name="Lander",
        component_path="../assets/lander/curated/lander_component.usda",
        xz=(1,1),
        scale=0.07
    ),

    Asset(
        name="Rover",
        component_path="../assets/rover/2/curated/rover2_component.usda",
        xz=(0.5,0.3),
        scale=0.05
    ),

    Asset(
        name="Habitat",
        component_path="../assets/hab_mod/curated/habmod_component.usda",
        xz=(-0.7,0.2),
        scale=0.15
    ),

    Asset(
        name="LandingPad",
        component_path="../assets/pad/curated/pad_component.usda",
        xz=(0,-0.8),
        scale=0.5
    )
]

assemble_scene_full(
    moon_component="../assets/moon/curated/moon_component.usda",
    assets=assets,
    output="lunar_scene.usda",
    moon_position=(0,0,0)
)

Opening: /home/ang/Desktop/scene/assets/moon/curated/moon_component.usda
Exists: True
Moon mesh loaded: 5856 vertices

Placing: Lander
Opening: /home/ang/Desktop/scene/assets/lander/curated/lander_component.usda
Exists: True
  world position: (1, 0.03151234417399202, 1)
  scale: 0.07

Placing: Rover
Opening: /home/ang/Desktop/scene/assets/rover/2/curated/rover2_component.usda
Exists: True
  world position: (0.5, -0.022661814292091263, 0.3)
  scale: 0.05

Placing: Habitat
Opening: /home/ang/Desktop/scene/assets/hab_mod/curated/habmod_component.usda
Exists: True
  world position: (-0.7, 0.18132680419201286, 0.2)
  scale: 0.15

Placing: LandingPad
Opening: /home/ang/Desktop/scene/assets/pad/curated/pad_component.usda
Exists: True
  world position: (0, 0.03245032703486217, -0.8)
  scale: 0.5

Scene written to: /home/ang/Desktop/scene/assembly/lunar_scene.usda


In [ ]:
import os

for root, dirs, files in os.walk("../assets"):
    for f in files:
        if f.endswith("_component.usda"):
            print(os.path.join(root, f))

../assets/hab_mod/curated/habmod_component.usda
../assets/moon/curated/moon_component.usda
../assets/rover/2/curated/rover2_component.usda
../assets/lander/curated_lander/lander_component.usda
../assets/lander/curated/lander_component.usda
../assets/pad/curated/pad_component.usda
